In [ ]:
import pyspark.sql.functions as F


silver_df = spark.table('nyc_taxi.silver.green_taxi')

duplicates = spark.table('nyc_taxi.quranatine.taxi_trips_duplicates_latest').withColumn(
    "trip_id",F.sha2(
        F.concat_ws("||", F.col("lpep_pickup_datetime"), F.col("lpep_dropoff_datetime"), F.col("PULocationID"),F.col("DOLocationID") ),256)
)

duplicates_ids = silver_df.select('trip_id').join(duplicates.select('trip_id'), 'trip_id', 'semi')

silver_df = silver_df.join(
    duplicates_ids, 
    'trip_id', 'left_anti'
    )
silver_df.mergeInto('nyc_taxi.silver.green_taxi','trip_id')
  .whenMatched().updateAll()
  .whenNotMatched().insertAll()
  .whenNotMatchedBySource().delete()
  .merge()